# PPE Detection – YOLO fine-tuning (4 classes)
**Do not edit any code.** Just run the cells in order (Shift+Enter on each, or Runtime → Run all).

Before starting: **Runtime → Change runtime type → T4 GPU → Save.**

Steps: 1) check GPU · 2) install · 3) choose your dataset ZIP · 4) train · 5) test + results · 6) download.

## Step 1 – Check the GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU OK:', torch.cuda.get_device_name(0))
else:
    print('NO GPU FOUND. Go to Runtime > Change runtime type > T4 GPU > Save, then run this cell again.')
    raise SystemExit('Stopped: GPU is required for training.')

## Step 2 – Install Ultralytics (YOLO)

In [ ]:
!pip install -q ultralytics
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

## Step 3 – Choose your dataset ZIP
Run the cell, then click **Choose files** and select `ppe_4classes.zip` from your computer. Wait until the upload reaches 100% (about 80 MB).

In [ ]:
import os, glob, zipfile, shutil

ZIP = '/content/ppe_4classes.zip'

# 1) Get the ZIP (skip the upload if it is already in /content)
if not os.path.exists(ZIP):
    from google.colab import files
    print('Click "Choose files" below and select ppe_4classes.zip ...')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file selected. Run this cell again and choose ppe_4classes.zip.')
    name = list(uploaded.keys())[0]
    if os.path.abspath(name) != ZIP:
        shutil.move(name, ZIP)

if not zipfile.is_zipfile(ZIP):
    os.remove(ZIP)
    raise SystemExit('That file is not a valid ZIP. Run this cell again and choose ppe_4classes.zip.')

# 2) Extract fresh
shutil.rmtree('/content/ds', ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/ds')

# 3) Find the dataset folder automatically (the one containing train/valid/test)
DS = None
for y in glob.glob('/content/ds/**/data.yaml', recursive=True):
    d = os.path.dirname(y)
    if all(os.path.isdir(f'{d}/{s}/images') for s in ('train', 'valid', 'test')):
        DS = d
        break
if DS is None:
    raise SystemExit('Could not find train/valid/test folders inside the ZIP. Make sure you chose ppe_4classes.zip.')

# 4) Verify image/label pairing and count annotations
expected = {'train': (521, 1385), 'valid': (114, 295), 'test': (82, 302)}
names = ['Hardhat_Worn', 'Hardhat_Not_Worn', 'Safety_Vest_Worn', 'Safety_Vest_Not_Worn']
all_ok = True
for s, (ei, ea) in expected.items():
    imgs = sorted(os.listdir(f'{DS}/{s}/images'))
    lbls = sorted(os.listdir(f'{DS}/{s}/labels'))
    stems_i = {os.path.splitext(f)[0] for f in imgs}
    stems_l = {os.path.splitext(f)[0] for f in lbls}
    n_ann, bad = 0, 0
    for f in lbls:
        for line in open(f'{DS}/{s}/labels/{f}').read().splitlines():
            if not line.strip():
                continue
            p = line.split()
            if len(p) != 5 or int(p[0]) not in range(4):
                bad += 1
            n_ann += 1
    ok = (len(imgs) == ei and n_ann == ea and stems_i == stems_l and bad == 0)
    all_ok &= ok
    print(f'{s:6s} images={len(imgs)} (expected {ei})  annotations={n_ann} (expected {ea})  '
          f'paired={stems_i == stems_l}  bad_lines={bad}  ->', 'OK' if ok else 'PROBLEM')
if not all_ok:
    raise SystemExit('Dataset check failed - see the PROBLEM line above and tell Claude.')

# 5) Write the YOLO data file with the correct absolute path (no manual editing needed)
with open('/content/data.yaml', 'w') as f:
    f.write(f'path: {DS}\ntrain: train/images\nval: valid/images\ntest: test/images\n\nnc: 4\nnames:\n')
    for i, n in enumerate(names):
        f.write(f'  {i}: {n}\n')
print('\nDataset verified and ready. data.yaml written.')

## Step 4 – Train (real fine-tuning from COCO-pretrained YOLO11s)
Takes roughly 20–40 minutes. The best model is chosen on the **validation** set. Do not close this tab.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')   # pretrained weights -> this is fine-tuning, not training from scratch
results = model.train(
    data='/content/data.yaml',
    epochs=100, patience=30, imgsz=640, batch=16,
    seed=0, deterministic=True, workers=2,
    save_period=10,
    project='/content/runs', name='ppe_yolo11s', exist_ok=True,
)
print('Training finished. Weights folder:', '/content/runs/ppe_yolo11s/weights')
print(os.listdir('/content/runs/ppe_yolo11s/weights'))

## Step 5 – Evaluate the chosen model (`best.pt`) on the TEST set, once

In [ ]:
import json, csv, os
from ultralytics import YOLO

best = YOLO('/content/runs/ppe_yolo11s/weights/best.pt')

v = best.val(data='/content/data.yaml', split='val', plots=False, verbose=False)
print('VALIDATION  P=%.3f R=%.3f mAP50=%.3f mAP50-95=%.3f' % (v.box.mp, v.box.mr, v.box.map50, v.box.map))

t = best.val(data='/content/data.yaml', split='test', plots=True,
             project='/content/results', name='test_eval', exist_ok=True)
print('TEST        P=%.3f R=%.3f mAP50=%.3f mAP50-95=%.3f' % (t.box.mp, t.box.mr, t.box.map50, t.box.map))

rows = [dict(split='test', cls='ALL', precision=t.box.mp, recall=t.box.mr, map50=t.box.map50, map50_95=t.box.map)]
for i, c in enumerate(t.box.ap_class_index):
    p, r, a50, a5095 = t.box.class_result(i)
    rows.append(dict(split='test', cls=best.names[int(c)], precision=p, recall=r, map50=a50, map50_95=a5095))
os.makedirs('/content/results', exist_ok=True)
json.dump(rows, open('/content/results/test_metrics.json', 'w'), indent=1)
with open('/content/results/test_metrics.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
print(open('/content/results/test_metrics.csv').read())

best.predict(f'{DS}/test/images',
             conf=0.25, save=True, project='/content/results', name='test_predictions', exist_ok=True)
print('Prediction images saved in /content/results/test_predictions')

## Step 6 – Download everything to your computer
A ZIP will download automatically. **Keep it – it is lost when this session closes.** Then send it (or the printed TEST line above) back to Claude.

In [ ]:
import shutil, os
from google.colab import files

out = '/content/PPE_outputs'
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out)
shutil.copytree('/content/runs/ppe_yolo11s', f'{out}/training_run')   # best.pt, last.pt, curves, results.csv, args.yaml
shutil.copytree('/content/results', f'{out}/results')                 # metrics CSV/JSON, confusion matrix, predictions
shutil.copy('/content/data.yaml', out)
shutil.make_archive('/content/PPE_outputs', 'zip', out)
print('ZIP size (MB):', round(os.path.getsize('/content/PPE_outputs.zip') / 1e6, 1))
files.download('/content/PPE_outputs.zip')